# SUV Fiyat Sınıflandırması: K-Means Algoritması

Bu notebook, SUV fiyat sınıflarını (Ekonomik, Orta, Yüksek, Premium) tahmin etmek için K-Means kümeleme algoritmasının uygulanmasını ve hiperparametre optimizasyonunu içermektedir.

## 1. Veri Yükleme

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, ConfusionMatrixDisplay,
                              classification_report, silhouette_score)
from sklearn.cluster import KMeans

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

RANDOM_STATE = 42

X_train = pd.read_csv('../data/X_train_final.csv')
X_test  = pd.read_csv('../data/X_test_final.csv')
y_train = pd.read_csv('../data/y_train_final.csv').iloc[:, 0]
y_test  = pd.read_csv('../data/y_test_final.csv').iloc[:, 0]

siniflar = ['Ekonomik', 'Orta', 'Yüksek', 'Premium']

print(f'Veri yüklendi. X_train: {X_train.shape}, X_test: {X_test.shape}')

## 2. Baseline Model (K=4)

Başlangıç aşamasında, hedef sınıf sayımız olan 4 küme ile temel K-Means modelini kuruyoruz. Gözetimsiz öğrenme sonucunda oluşan kümeleri, eğitim setindeki en yoğun sınıflarla eşleştiriyoruz.

In [ ]:
baseline_model = KMeans(n_clusters=4, random_state=RANDOM_STATE, n_init=10)
train_clusters_base = baseline_model.fit_predict(X_train)

def get_cluster_mapping(clusters, y_true):
    mapping = {}
    for c in np.unique(clusters):
        mask = (clusters == c)
        mode_res = y_true[mask].mode()
        mapping[c] = mode_res[0] if not mode_res.empty else y_true.mode()[0]
    return mapping

baseline_mapping = get_cluster_mapping(train_clusters_base, y_train)
test_clusters_base = baseline_model.predict(X_test)
y_pred_base = np.array([baseline_mapping[c] for c in test_clusters_base])

print('=== BASELINE SONUÇLARI ===')
print(f'F1 macro : {f1_score(y_test, y_pred_base, average="macro"):.4f}')

## 3. Hiperparametre Optimizasyonu (Tuning)

En uygun küme sayısını belirlemek için Elbow (Dirsek) ve Silhouette skorlarını analiz ediyoruz.

In [ ]:
inertia = []
silhouette_scores = []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_train)
    inertia.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_train, labels))

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(K_range, inertia, 'bo-', label='Inertia (Elbow)')
ax1.set_xlabel('Küme Sayısı (K)')
ax1.set_ylabel('Inertia', color='b')
ax2 = ax1.twinx()
ax2.plot(K_range, silhouette_scores, 'ro-', label='Silhouette Score')
ax2.set_ylabel('Silhouette Score', color='r')
plt.title('K-Means Tuning: Elbow ve Silhouette Analizi')
plt.show()

### Final Model

Analizler sonucunda elde edilen en iyi parametrelerle final modelimizi yapılandırıyoruz.

In [ ]:
best_k = 4 
tuned_model = KMeans(n_clusters=best_k, init='k-means++', n_init=20, random_state=RANDOM_STATE)
train_clusters_tuned = tuned_model.fit_predict(X_train)
tuned_mapping = get_cluster_mapping(train_clusters_tuned, y_train)
test_clusters_tuned = tuned_model.predict(X_test)
y_pred_tuned = np.array([tuned_mapping[c] for c in test_clusters_tuned])

print(f'=== TUNED SONUÇLARI (K={best_k}) ===')
print(f'F1 macro : {f1_score(y_test, y_pred_tuned, average="macro"):.4f}')

## 4. Karşılaştırma ve Değerlendirme

In [ ]:
metrics = ['Accuracy', 'F1-macro', 'Precision (macro)', 'Recall (macro)']
karsilastirma = pd.DataFrame({
    'Metrik': metrics,
    'Baseline': [
        accuracy_score(y_test, y_pred_base),
        f1_score(y_test, y_pred_base, average='macro'),
        precision_score(y_test, y_pred_base, average='macro'),
        recall_score(y_test, y_pred_base, average='macro'),
    ],
    'Tuned': [
        accuracy_score(y_test, y_pred_tuned),
        f1_score(y_test, y_pred_tuned, average='macro'),
        precision_score(y_test, y_pred_tuned, average='macro'),
        recall_score(y_test, y_pred_tuned, average='macro'),
    ],
})
print(karsilastirma.round(4).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, y_p, title in zip(axes, [y_pred_base, y_pred_tuned], ['Baseline', 'Tuned']):
    cm = confusion_matrix(y_test, y_p, labels=siniflar)
    disp = ConfusionMatrixDisplay(cm, display_labels=siniflar)
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(title)
plt.tight_layout()
plt.show()

## 5. Yorum

### 5.1 Algoritmanın çalışma mantığı
K-Means, veriyi benzer özelliklere sahip K adet kümeye ayıran bir gözetimsiz öğrenme algoritmasıdır. Sınıflandırma için kullanıldığında, her küme içindeki en yoğun sınıf etiketi o kümenin temsili etiketi olarak atanır.

### 5.2 Tuning sonuçları
Elbow grafiğinde bükülme noktası ve silhouette skorlarına göre en uygun küme sayısı belirlenmiştir. Kasko bedeli ve yıl gibi veriler kümelemede belirleyici olmuştur.

### 5.3 Sınıf bazlı performans
Özellikle 'Premium' ve 'Ekonomik' sınıflar uç değerler olduğu için daha iyi ayrışırken, 'Orta' ve 'Yüksek' sınıflar birbirine yakın özellikler nedeniyle karıştırılabilmektedir.

### 5.4 Bu veriye uygun mu?
K-Means aslında bir kümeleme algoritması olduğu için sınıflandırma başarısı gözetimli modellere göre daha düşük kalabilir. Ancak veri setindeki doğal gruplanmaları görmek adına faydalıdır.